# Phase 2 — Feature Engineering

## Overview
Raw macro series are not directly usable as model inputs. This notebook transforms them into a **40-column feature matrix** that captures the economic mechanisms driving inflation.

### Feature groups and economic rationale

| Group | Features | Why they matter |
|-------|----------|-----------------|
| **Lag features** | CPI(t-1), (t-3), (t-6), (t-12) | Inflation is autocorrelated — past values predict future values |
| **Rolling statistics** | 3-month / 12-month rolling mean & std of CPI | Trend smoothing and volatility signal |
| **Rate-of-change** | MoM and YoY % change for M2, unemployment, oil, PPI | Acceleration matters more than level |
| **Interaction terms** | M2 growth × rate differential, yield curve spread | Monetarist and Keynesian composite signals |
| **Seasonal dummies** | month_1 … month_12 | CPI has well-documented seasonal patterns (e.g. January resets) |
| **Regime features** | post_covid (2020+) | COVID-era structural break changes all relationships |

### Data leakage prevention
All lags use **strictly past values**. The scaler is fit on training data only and applied to test data. `TimeSeriesSplit` is used for all cross-validation — no shuffling of temporal data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
from src.ingest import load_raw
from src.features import build_feature_matrix, split, get_xy, save_processed

raw = load_raw()
df = build_feature_matrix(raw)
print(f'Feature matrix: {df.shape}')
df.head(3)

## 1. Build Feature Matrix

`build_feature_matrix()` applies all transformations in a single pass:
- Resamples all series to monthly frequency (month-start)
- Forward-fills GDP (quarterly → monthly)
- Computes YoY inflation rate as the target variable
- Engineers all lag, rolling, interaction, seasonal, and regime features

The result is a **single flat DataFrame** where every row is a complete monthly observation ready for supervised learning.

## 2. Missing Value Check

Some NaN values are expected and acceptable:
- The first 12 rows will be NaN for YoY features (need 12 months of history)
- The first 12 rows will be NaN for CPI(t-12) lag
- Rolling 12-month stats lose the first 12 rows

`dropna()` removes all incomplete rows. The final dataset starts around 1991 despite raw data beginning in 1990.

## 3. Train / Test Split

We use a **strict temporal split** — no shuffling:
- **Train**: 1991–2018 (~27 years, ~320 months)
- **Test**: 2019–2025 (~6 years, ~80 months)

This is consistent with the walk-forward evaluation in Phase 4. The test period deliberately includes COVID-era inflation (2021–2022), the hardest forecasting environment in decades.

## 4. Feature / Target Extraction

`get_xy()` separates features from the target (`inflation_rate`) and drops raw CPI columns to prevent data leakage (the model should never see the raw CPI level that directly encodes the target).

## 5. Feature–Target Correlations

This chart ranks all features by their linear correlation with inflation rate on the **training set only**. Key findings:
- CPI lag features (especially lag-1 and lag-12) dominate — strong autocorrelation signal
- Rolling means are near-perfect proxies of recent inflation — high signal but risk of leakage if not lagged correctly (they are shifted by 1 in our pipeline)
- Among exogenous variables, PPI and oil show the strongest correlation — supporting cost-push theory

## 6. Lag Feature Scatterplots

These scatterplots show the raw relationship between each CPI lag and the inflation target:
- **Lag-1** has a nearly linear relationship — very high predictive power for the next month
- **Lag-12** shows a tighter band — annual CPI levels predict annual inflation well
- Non-linear clusters visible at high-inflation values (2022 outliers) motivate the use of XGBoost over pure linear models

## 7. Save Processed Data

The final feature matrix is saved to `data/processed/features.csv` with the date index preserved. This file is the single source of truth for all downstream model training.

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isna().sum()[df.isna().sum() > 0])
print('Total:', df.isna().sum().sum())

In [ ]:
# Train / test split
train, test = split(df)
print(f'Train: {len(train)} rows  ({train.index[0].date()} – {train.index[-1].date()})')
print(f'Test:  {len(test)} rows  ({test.index[0].date()} – {test.index[-1].date()})')

In [ ]:
# Feature / target extraction
X_train, y_train = get_xy(train)
X_test,  y_test  = get_xy(test)
print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}  | y_test:  {y_test.shape}')

In [ ]:
# Correlation of features with target
corr = X_train.corrwith(y_train).abs().sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(8, 6))
corr.plot.barh(ax=ax, color='steelblue')
ax.set_title('Top 20 Features by Absolute Correlation with Inflation Rate')
ax.set_xlabel('|Correlation|')
plt.tight_layout()
plt.show()

In [ ]:
# Lag feature visualisation
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, lag in zip(axes.flatten(), [1, 3, 6, 12]):
    ax.scatter(train[f'cpi_lag{lag}'], y_train, alpha=0.3, s=10, color='steelblue')
    ax.set_xlabel(f'CPI lag {lag}')
    ax.set_ylabel('Inflation Rate')
    ax.set_title(f'CPI lag {lag} vs Inflation')
plt.suptitle('Lag Feature Scatterplots')
plt.tight_layout()
plt.show()

In [ ]:
# Save processed feature matrix
save_processed(df)
print('Feature matrix saved.')